In [21]:
# @title Служебная ячейка: импорты и функции проверки { display-mode: "form" }
import inspect
import re
import time

import numpy as np
import pandas as pd

SEED = 0
rng = np.random.default_rng(SEED)


def assert_no_loops(func):
    """Проверяет, что в теле функции нет циклов for/while (включения списков тоже считаются)."""
    source = inspect.getsource(func)
    body = source.split('"""')[-1] if '"""' in source else source
    assert not re.search(r"\b(for|while)\b", body), (
        f"В функции {func.__name__} есть цикл. Здесь нужно решение без циклов — операциями над массивами."
    )


print("Готово")

Готово


# Часть 1. Python

Задание 1.1. Включения списков и срезы

In [22]:
def squares_of_evens(xs):
    return [x**2 for x in xs if x % 2 == 0][::-1]

assert squares_of_evens([1, 2, 3, 4, 6]) == [36, 16, 4], "Для [1, 2, 3, 4, 6] ожидается [36, 16, 4]"
assert squares_of_evens([]) == [], "Для пустого списка ожидается пустой список"
assert squares_of_evens([1, 3, 5]) == [], "Если чётных нет, ожидается пустой список"
assert squares_of_evens([-2, 0]) == [0, 4], "Отрицательные и ноль — тоже чётные; порядок — обратный"
assert isinstance(squares_of_evens(range(5)), list), "Функция должна возвращать список и принимать любой итерируемый объект"
print("OK")

OK


Задание 1.2. Словари и подсчёт

In [23]:
def word_counts(text):
    res = text.lower().split()
    return {word: res.count(word) for word in set(res)}

assert word_counts("a b A") == {"a": 2, "b": 1}, "Регистр не должен учитываться: 'A' и 'a' — одно слово"
assert word_counts("") == {}, "Для пустой строки ожидается пустой словарь"
assert word_counts("мама мыла раму мама") == {"мама": 2, "мыла": 1, "раму": 1}, "Неверный подсчёт для 'мама мыла раму мама'"
print("OK")

OK


Задание 1.3. Ссылки и изменяемые аргументы

In [24]:
def add_item(item, bucket=None):
    if (bucket == None):
        bucket = []

    bucket.append(item)

    return bucket

assert add_item(1) == [1], "add_item(1) должен вернуть [1]"
assert add_item(2) == [2], "Второй вызов без bucket вернул не [2]: список по умолчанию общий для всех вызовов"
b = [0]
assert add_item(5, b) is b and b == [0, 5], "Переданный список должен дополняться и возвращаться тот же объект"
print("OK")

OK


Задание 1.4. zip, sorted и key

In [25]:
def top_k(names, scores, k):
    return [name for name, score in sorted(zip(names, scores), key=lambda x: x[1], reverse=True)][:2]

assert top_k(["a", "b", "c"], [1, 3, 2], 2) == ["b", "c"], "Для баллов [1, 3, 2] два лучших — b и c"
assert top_k(["a", "b", "c"], [5, 5, 1], 2) == ["a", "b"], "При равных баллах порядок исходного списка должен сохраняться"
assert top_k(["a"], [1], 5) == ["a"], "Если k больше числа имён, вернуть все имена"
print("OK")

OK


# Часть 2. NumPy

In [26]:
x = rng.normal(size=1_000_000)

start = time.perf_counter()
total = 0.0
for v in x:
    total += v * v
loop_time = time.perf_counter() - start

start = time.perf_counter()
total_np = np.sum(x * x)
numpy_time = time.perf_counter() - start

print(f"цикл: {loop_time:.3f} с, NumPy: {numpy_time:.4f} с, ускорение в {loop_time / numpy_time:.0f} раз")

цикл: 0.689 с, NumPy: 0.0027 с, ускорение в 251 раз


Задание 2.1. Агрегаты по осям и broadcasting: стандартизация

In [27]:
def standardize(X):
    return (X - X.mean(axis=0)) / X.std(axis=0)

X_test = rng.normal(loc=[1, 100, -5], scale=[1, 50, 0.1], size=(200, 3))
X_copy = X_test.copy()
Z = standardize(X_test)
assert Z.shape == X_test.shape, f"Форма результата {Z.shape}, а должна быть {X_test.shape}"
assert np.allclose(Z.mean(axis=0), 0), "Средние столбцов после стандартизации должны быть равны 0"
assert np.allclose(Z.std(axis=0), 1), "Стандартные отклонения столбцов должны быть равны 1 (а не строк!)"
assert np.array_equal(X_test, X_copy), "Функция изменила исходный массив"
assert_no_loops(standardize)
print("OK")

OK


Задание 2.2. Маски и копии

In [28]:
def clip_negative(X):
    local_X = X.copy()
    local_X[local_X < 0] = 0

    return local_X, np.sum(X < 0)

A = np.array([[1.0, -2.0], [-3.0, 4.0]])
A_copy = A.copy()
Y, count = clip_negative(A)
assert np.array_equal(Y, [[1, 0], [0, 4]]), f"Ожидалось [[1, 0], [0, 4]], получено {Y.tolist()}"
assert count == 2, f"Отрицательных элементов 2, а функция вернула {count}"
assert np.array_equal(A, A_copy), "Функция изменила исходный массив: срез или маска без .copy() — это тот же массив"
assert_no_loops(clip_negative)
print("OK")

OK


Задание 2.3. Индексация массивами: one-hot

In [29]:
def one_hot(y, K):
    matrix = np.zeros((len(y), K))
    matrix[np.arange(len(y)), y] = 1

    return matrix

E = one_hot(np.array([2, 0, 1, 2]), 3)
assert E.shape == (4, 3), f"Форма должна быть (4, 3), получено {E.shape}"
assert np.array_equal(E, [[0, 0, 1], [1, 0, 0], [0, 1, 0], [0, 0, 1]]), f"Неверная матрица: {E.tolist()}"
assert np.array_equal(one_hot(np.array([0]), 1), [[1]]), "Для y=[0], K=1 ожидается [[1]]"
assert_no_loops(one_hot)
print("OK")

OK


Задание 2.4. Попарные расстояния без циклов

In [30]:
def pairwise_sq_dists(A, B):
    matrix = np.zeros((A.shape[0], B.shape[0]))

    A_dots = (A**2).sum(axis=1)[:, np.newaxis]
    B_dots = (B**2).sum(axis=1)
    pair = A @ B.T

    raw_dist = A_dots + B_dots - 2 * pair

    matrix[:] = np.maximum(raw_dist, 0)

    return matrix

A = rng.normal(size=(7, 4))
B = rng.normal(size=(5, 4))
D = pairwise_sq_dists(A, B)
reference = np.array([[np.sum((a - b) ** 2) for b in B] for a in A])
assert D.shape == (7, 5), f"Форма должна быть (7, 5), получено {D.shape}"
assert np.allclose(D, reference), "Расстояния не совпадают с посчитанными в лоб"
assert np.allclose(np.diag(pairwise_sq_dists(A, A)), 0), "Расстояние от точки до самой себя должно быть 0"
assert (pairwise_sq_dists(A, A) >= 0).all(), "Квадраты расстояний не могут быть отрицательными"
assert_no_loops(pairwise_sq_dists)
print("OK")

OK


Задание 2.5. Сортировка по строкам: k ближайших

In [31]:
def k_nearest(D, k):
    return np.argsort(D, axis=1)[:,:k]

D = np.array([[3.0, 1.0, 2.0, 0.5], [0.0, 9.0, 8.0, 7.0]])
idx = k_nearest(D, 2)
assert idx.shape == (2, 2), f"Форма должна быть (2, 2), получено {np.shape(idx)}"
assert np.array_equal(idx, [[3, 1], [0, 3]]), f"Ожидалось [[3, 1], [0, 3]], получено {np.asarray(idx).tolist()}"
assert np.array_equal(k_nearest(D, 4)[1], [0, 3, 2, 1]), "Индексы должны идти по возрастанию расстояния"
assert_no_loops(k_nearest)
print("OK")

OK


# Часть 3. pandas

In [32]:
from sklearn.datasets import fetch_openml

titanic = fetch_openml(data_id=40945, as_frame=True, parser="auto").frame
# boat и body почти однозначно выдают ответ (номер шлюпки, номер найденного тела) — убираем.
titanic = titanic.drop(columns=["boat", "body", "home.dest", "ticket", "cabin"])
titanic["survived"] = titanic["survived"].astype(int)
titanic["sex"] = titanic["sex"].astype(str)
titanic["embarked"] = titanic["embarked"].astype(object)
print(titanic.shape)
titanic.head()

(1309, 9)


,pclass,survived,name,sex,age,sibsp,parch,fare,embarked
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,211.3375,S
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,151.5500,S
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,151.5500,S
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,151.5500,S
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,151.5500,S


Задание 3.1. Пропуски

In [33]:
def missing_report(df):
    rep = df.isna().sum()
    rep = rep[rep > 0]

    return rep.sort_values(ascending=False)

report = missing_report(titanic)
assert isinstance(report, pd.Series), "Нужно вернуть pd.Series"
assert list(report.index) == ["age", "embarked", "fare"], f"Ожидались столбцы age, embarked, fare по убыванию, получено {list(report.index)}"
assert report["age"] == 263, f"Пропусков в age 263, получено {report['age']}"
assert missing_report(pd.DataFrame({"a": [1, 2]})).empty, "Если пропусков нет, результат должен быть пустым"
print("OK")

OK


Задание 3.2. Фильтрация и группировка

In [34]:
def survival_by_group(df):
    clean_df = df.dropna(subset=["age"])
    adults = clean_df[clean_df["age"] >= 18]

    return adults.groupby(["sex", "pclass"])["survived"].mean()

rates = survival_by_group(titanic)
assert isinstance(rates, pd.Series), "Нужно вернуть pd.Series"
assert rates.index.names == ["sex", "pclass"], f"Индекс должен быть (sex, pclass), получено {rates.index.names}"
assert len(rates) == 6, f"Групп (пол × класс) должно быть 6, получено {len(rates)}"
assert abs(rates[("female", 1)] - 0.968) < 1e-3, "Неверная доля выживших для женщин 1 класса: проверьте фильтр по возрасту"
assert abs(rates[("male", 3)] - 0.1557) < 1e-3, "Неверная доля выживших для мужчин 3 класса"
print("OK")

OK


Задание 3.3. Заполнение пропусков по группам

In [35]:
def fill_age(df):
    local_df = df.copy()
    group = local_df.groupby(["sex", "pclass"])["age"].transform("median")

    local_df["age"] = local_df["age"].fillna(group)

    return local_df

before = titanic["age"].isna().sum()
filled = fill_age(titanic)
assert titanic["age"].isna().sum() == before, "Функция изменила исходную таблицу — работайте с копией (df.copy())"
assert filled["age"].isna().sum() == 0, "В age остались пропуски"
known = titanic["age"].notna()
assert (filled.loc[known, "age"] == titanic.loc[known, "age"]).all(), "Известные значения возраста не должны меняться"
i = titanic.index[titanic["age"].isna() & (titanic["sex"] == "male") & (titanic["pclass"] == 3)][0]
expected = titanic.loc[(titanic["sex"] == "male") & (titanic["pclass"] == 3), "age"].median()
assert filled.loc[i, "age"] == expected, "Пропуск у мужчины 3 класса должен заполниться медианой возраста мужчин 3 класса"
print("OK")

OK


Задание 3.4. Признаки для модели

In [36]:
def make_features(df):
    local_df = fill_age(df)
    local_df["fare"] = local_df["fare"].fillna(local_df["fare"].median())
    local_df["embarked"] = local_df["embarked"].fillna(local_df["embarked"].mode()[0])

    local_df["family"] = local_df["sibsp"] + local_df["parch"] + 1

    Y = local_df["survived"].to_numpy().astype(int)

    features = ["pclass", "sex", "age", "fare", "family", "embarked"]
    X_df = local_df[features]
    X_df = pd.get_dummies(X_df, columns=["sex", "embarked"])

    X = X_df.to_numpy().astype(float)

    feature_names = list(X_df.columns)

    return X, Y, feature_names

X, y, names = make_features(titanic)
assert isinstance(X, np.ndarray) and X.dtype == float, "X должен быть массивом NumPy типа float"
assert X.shape == (1309, 9), f"Ожидалась форма (1309, 9): pclass, age, fare, family + 2 столбца пола + 3 порта; получено {X.shape}"
assert not np.isnan(X).any(), "В X остались пропуски"
assert len(names) == X.shape[1], "Число имён признаков должно совпадать с числом столбцов X"
assert "family" in names and "sex_male" in names, f"Среди признаков должны быть family и sex_male, получено {names}"
assert y.shape == (1309,) and set(np.unique(y)) == {0, 1}, "y должен быть вектором из 0 и 1 длины 1309"
j = names.index("family")
assert X[0, j] == titanic.loc[0, "sibsp"] + titanic.loc[0, "parch"] + 1, "Признак family посчитан неверно"
print("OK")

OK


Задание 3.5. Вопросы
1.   Почему в заданиях 2.2 и 3.3 пришлось явно делать копию, а в задании 2.1 — нет?
2.   Во сколько раз векторизованное вычисление было быстрее цикла в начале части 2? Откуда такая разница?
3.   Почему пол и порт посадки кодируются one-hot, а класс билета можно оставить числом?

Ответы
1.   Потому что присваивание в питоне работает как создание поинтера на конкретную ячейку в памяти. Это приводит к тому, что изменения по одному поинтеру приведут к изменению данный в этой ячейке, что напрямую испортит данные по другому поинтеру. В задание 2.1 создание новой ячейки памяти происходит неявно и автоматически в результате арифметической операции, из-за чего целостность исходных данных не нарушается
2.   В 251 раз. Компилятор питона обходит AST исходного кода один раз и генерирует промежуточный байт-код. В рантайме PVM вынуждена выполнять the interpreter loop -- гигантский цикл switch-case на C. На каждой из 1 000 000 итераций рантайм тратит процессорные такты на декодирование инструкций байт-кода (FOR_ITER, BINARY_OP), управление внутренним стеком PVM и ветвление. Для процессора этот бесконечный switch оборачивается постоянными Branch Mispredictions, или же ошибками предсказания переходов, из-за чего конвейер процессора просто стоит на месте. Выражение np.sum(x * x) в свою очередь с точки зрения компилятора питона -- это AST, которое в разы меньше аналогичного на пайтон и которое транслируется всего в пару инструкций байт-кода. Интерпретатор полностью передает управление скомпилированному нативно C-коду NumPy. На уровне процессора запускается цикл, который работает напрямую на железе, минуя какие-либо виртуальные машины и накладные расходы на интерпретацию. Так же цикл, написанный на пайтон, будет постоянно совершать кэш-промахи, что заставляет процессор простаивать. В NumPy происходит аппаратная оптимизация с учетом особенностей архитектуры, на которой код компилируется и запускается
3.   Класс билета -- ordinal признак, а пол и порт -- nominal. В первом случае модель из-за того, что данные представлены числами, четко может считать изменение 1->2->3 и установить какую-то сзявь. Во втором случае данные представлены в виде текста(условно, нельзя сказать, что пол "female" в качестве значения признака лучше или хуже, чем "male")